In [1]:
!pip install streamlit pyjwt bcrypt python-dotenv pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 23.3 MB/s eta 0:00:00


In [2]:
%%writefile db.py
import sqlite3
import bcrypt
import datetime
import time

DB_NAME = "users.db"
max_login_attempts = 3
lockout_time = 300  # 5 minutes in seconds

def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    # Users table
    c.execute('''CREATE TABLE IF NOT EXISTS users
                 (email TEXT PRIMARY KEY, password BLOB, created_at TEXT)''')

    # Password History table
    c.execute('''CREATE TABLE IF NOT EXISTS password_history
                 (id INTEGER PRIMARY KEY AUTOINCREMENT,
                  email TEXT,
                  password BLOB,
                  set_at TEXT,
                  FOREIGN KEY(email) REFERENCES users(email))''')

    # Login Attempts table
    c.execute('''CREATE TABLE IF NOT EXISTS login_attempts
                 (email TEXT PRIMARY KEY, attempts INTEGER, last_attempt REAL)''')

    conn.commit()
    conn.close()

    # Ensure Admin Exists
    init_admin()

def init_admin():
    if not check_user_exists("admin@llm.com"):
        register_user("admin@llm.com", "Admin@123")
        print("Admin account created.")

def _get_timestamp():
    return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

# --- Rate Limiting ---
def get_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data if data else (0, 0)

def increment_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    attempts, _ = get_login_attempts(email)
    new_attempts = attempts + 1
    now = time.time()

    c.execute("INSERT OR REPLACE INTO login_attempts (email, attempts, last_attempt) VALUES (?, ?, ?)",
              (email, new_attempts, now))
    conn.commit()
    conn.close()

def reset_login_attempts(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()

def is_rate_limited(email):
    attempts, last_attempt = get_login_attempts(email)
    if attempts >= max_login_attempts:
        if time.time() - last_attempt < lockout_time:
            return True, lockout_time - (time.time() - last_attempt)
        else:
            # Lockout expired
            reset_login_attempts(email)
    return False, 0

# --- User Management ---
def register_user(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    try:
        salt = bcrypt.gensalt()
        hashed = bcrypt.hashpw(password.encode('utf-8'), salt)
        now = _get_timestamp()

        c.execute("INSERT INTO users (email, password, created_at) VALUES (?, ?, ?)", (email, hashed, now))
        c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))

        conn.commit()
        return True
    except sqlite3.IntegrityError:
        return False
    finally:
        conn.close()

def authenticate_user(email, password):
    # Check rate limit first in app.py or here? Better to check before auth logic.
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    if data:
        stored_hash = data[0]
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            reset_login_attempts(email) # Success reset
            return True

    # Failed
    increment_login_attempts(email)
    return False

def check_is_old_password(email, password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password, set_at FROM password_history WHERE email = ? ORDER BY set_at DESC", (email,))
    history = c.fetchall()
    conn.close()

    for stored_hash, set_at in history:
        if bcrypt.checkpw(password.encode('utf-8'), stored_hash):
            return set_at
    return None

def check_password_reused(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT password FROM password_history WHERE email = ?", (email,))
    history = c.fetchall()
    conn.close()

    for (stored_hash,) in history:
        if bcrypt.checkpw(new_password.encode('utf-8'), stored_hash):
            return True
    return False

def check_user_exists(email):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT 1 FROM users WHERE email = ?", (email,))
    data = c.fetchone()
    conn.close()
    return data is not None

def update_password(email, new_password):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()

    salt = bcrypt.gensalt()
    hashed = bcrypt.hashpw(new_password.encode('utf-8'), salt)
    now = _get_timestamp()

    c.execute("UPDATE users SET password = ? WHERE email = ?", (hashed, email))
    c.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now))

    conn.commit()
    conn.close()

# --- Admin Functions ---
def get_all_users():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT email, created_at FROM users")
    data = c.fetchall()
    conn.close()
    return data

def delete_user(email):
    if email == "admin@llm.com": return False # Protect admin
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("DELETE FROM users WHERE email = ?", (email,))
    c.execute("DELETE FROM password_history WHERE email = ?", (email,))
    c.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    conn.commit()
    conn.close()
    return True

Writing db.py


In [3]:
%%writefile app.py
import os, re, sqlite3, jwt, bcrypt, datetime, time, secrets, smtplib, streamlit as st
from email.utils import formatdate, make_msgid
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# 🌸 1. Force Streamlit Soft Pink Theme
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml", "w") as f:
    f.write('[theme]\nbase="light"\nprimaryColor="#D98BA3"\nbackgroundColor="#FFFAFC"\nsecondaryBackgroundColor="#FDEFF4"\ntextColor="#3D3036"\n')

st.set_page_config(page_title="Secure Password Reset", page_icon="🌸", layout="wide")

COLORS = {
    "primary": "#F8C8DC",      # Soft Baby Pink
    "bg_main": "#FFFAFC",      # Off-White
    "secondary": "#F4B6C2",    # Blush Pink
    "accent": "#D98BA3",       # Dusty Rose
    "bg_card": "#FFFFFF",      # Card White
    "text_main": "#3D3036",    # Dark Charcoal
    "text_muted": "#7A6770",   # Muted charcoal (readable on white)
    "lavender": "#EDE4F7"      # Light Lavender
}

JWT_SECRET = os.getenv("JWT_SECRET", "super-secret-otp-key-2026")
SENDER_EMAIL = "ranamehak570@gmail.com"  # 🚀 Your sender email address
EMAIL_PASSWORD = os.getenv("EMAIL_PASSWORD")  # 🚀 Gmail App Password for ranamehak570@gmail.com (from Colab Secrets)
OTP_EXPIRY_MINUTES = 5

# 🌸 2. Soft Pink Pastel CSS (navbar, background shapes, cards, buttons, animations)
st.markdown(f"""
<style>
    @import url('https://fonts.googleapis.com/css2?family=Poppins:wght@400;500;600;700&family=Inter:wght@300;400;500;600&display=swap');

    html {{ scroll-behavior: smooth; }}
    html, body, .stApp {{
        font-family: 'Inter', sans-serif !important;
        color: {COLORS['text_main']} !important;
    }}
    .stApp {{ background: linear-gradient(180deg, {COLORS['bg_main']} 0%, #FFF3F8 100%) !important; }}
    header, footer, #MainMenu {{ visibility: hidden !important; display: none !important; }}

    /* Blurred pastel decorative circles */
    .stApp::before {{
        content: ""; position: fixed; top: -140px; right: -120px; width: 440px; height: 440px;
        border-radius: 50%; background: {COLORS['primary']}; filter: blur(90px); opacity: 0.55;
        z-index: 0; pointer-events: none;
    }}
    .stApp::after {{
        content: ""; position: fixed; bottom: -160px; left: -140px; width: 460px; height: 460px;
        border-radius: 50%; background: {COLORS['lavender']}; filter: blur(95px); opacity: 0.8;
        z-index: 0; pointer-events: none;
    }}

    .block-container {{ position: relative; z-index: 1; padding: 6rem 2rem 2rem !important; max-width: 1100px; }}

    /* Headings */
    h1, h2, h3, h4 {{ font-family: 'Poppins', sans-serif !important; color: {COLORS['text_main']} !important; }}
    label p {{ font-weight: 600 !important; color: {COLORS['text_main']} !important; }}

    /* Navigation bar */
    .nav-bar {{
        position: fixed; top: 0; left: 0; right: 0; height: 64px; z-index: 999;
        display: flex; align-items: center; justify-content: space-between; padding: 0 2.5rem;
        background: rgba(255, 255, 255, 0.78); backdrop-filter: blur(12px); -webkit-backdrop-filter: blur(12px);
        border-bottom: 1px solid {COLORS['primary']};
    }}
    .nav-brand {{ font-family: 'Poppins', sans-serif; font-weight: 700; font-size: 18px; color: {COLORS['text_main']}; }}
    .nav-links {{ display: flex; gap: 1.5rem; }}
    .nav-links a {{
        color: {COLORS['text_main']}; text-decoration: none; font-weight: 500; font-size: 14px;
        padding: 6px 14px; border-radius: 999px; transition: all 0.25s ease;
    }}
    .nav-links a:hover {{ background: {COLORS['primary']}; color: {COLORS['text_main']}; }}
    .nav-links a:focus-visible {{ outline: 3px solid {COLORS['accent']}; outline-offset: 2px; }}

    /* Hero */
    .hero {{ text-align: center; padding: 1.5rem 0 2rem; animation: fadeUp 0.7s ease both; }}
    .hero-badge {{
        display: inline-block; background: {COLORS['lavender']}; color: {COLORS['text_main']};
        font-size: 13px; font-weight: 600; padding: 6px 16px; border-radius: 999px; margin-bottom: 14px;
        border: 1px solid {COLORS['primary']};
    }}
    .hero h1 {{ font-size: 2.6rem !important; margin: 0 0 8px !important; font-weight: 700 !important; }}
    .hero p {{ color: {COLORS['text_muted']}; font-size: 16px; margin: 0 auto; max-width: 520px; line-height: 1.6; }}

    /* Main card (Streamlit bordered container) */
    div[data-testid="stVerticalBlockBorderWrapper"] {{
        background-color: {COLORS['bg_card']} !important; border: 1.5px solid {COLORS['secondary']} !important;
        border-radius: 22px !important; padding: 28px !important;
        box-shadow: 0 12px 32px rgba(217, 139, 163, 0.16) !important;
        transition: transform 0.3s ease, box-shadow 0.3s ease; animation: fadeUp 0.8s ease both;
    }}
    div[data-testid="stVerticalBlockBorderWrapper"]:hover {{
        transform: translateY(-3px); box-shadow: 0 18px 40px rgba(217, 139, 163, 0.24) !important;
    }}

    /* Inputs */
    div[data-baseweb="base-input"] {{ background-color: transparent !important; border: none !important; }}
    div[data-baseweb="input"] {{
        background-color: {COLORS['bg_card']} !important; border: 1.5px solid {COLORS['secondary']} !important;
        border-radius: 12px !important; transition: all 0.25s ease;
    }}
    div[data-baseweb="input"]:hover {{ border-color: {COLORS['accent']} !important; }}
    div[data-baseweb="input"]:focus-within {{
        border-color: {COLORS['accent']} !important; box-shadow: 0 0 0 4px rgba(244, 182, 194, 0.45) !important;
    }}
    input {{
        background: transparent !important; color: {COLORS['text_main']} !important;
        -webkit-text-fill-color: {COLORS['text_main']} !important; font-size: 16px !important;
    }}
    input::placeholder {{ color: #A8949D !important; -webkit-text-fill-color: #A8949D !important; }}

    /* Buttons */
    div[data-testid="stButton"] button {{
        background-color: {COLORS['primary']} !important; color: {COLORS['text_main']} !important;
        border: 1.5px solid {COLORS['secondary']} !important; border-radius: 12px !important;
        font-family: 'Inter', sans-serif !important; font-weight: 600 !important; font-size: 15px !important;
        height: 48px !important; min-height: 48px !important; white-space: nowrap !important;
        display: flex !important; align-items: center !important; justify-content: center !important;
        box-shadow: 0 4px 12px rgba(217, 139, 163, 0.25) !important; width: 100%; transition: all 0.25s ease !important;
    }}
    div[data-testid="stButton"] button:hover {{
        background-color: {COLORS['accent']} !important; border-color: {COLORS['accent']} !important;
        color: #FFFFFF !important; transform: translateY(-2px) !important;
        box-shadow: 0 8px 18px rgba(217, 139, 163, 0.35) !important;
    }}
    div[data-testid="stButton"] button:focus-visible {{ outline: 3px solid {COLORS['accent']} !important; outline-offset: 2px !important; }}

    /* Alerts */
    div[data-testid="stAlert"] {{ border-radius: 12px !important; border: 1px solid {COLORS['primary']} !important; }}

    /* How it works */
    .section-title {{ text-align: center; font-family: 'Poppins', sans-serif; font-size: 1.4rem; font-weight: 600; margin: 3.5rem 0 1.2rem; }}
    .steps {{ display: grid; grid-template-columns: repeat(auto-fit, minmax(220px, 1fr)); gap: 18px; animation: fadeUp 1s ease both; }}
    .step-card {{
        background: {COLORS['bg_card']}; border: 1.5px solid {COLORS['primary']}; border-radius: 18px;
        padding: 22px; text-align: center; transition: all 0.3s ease; box-shadow: 0 6px 18px rgba(217, 139, 163, 0.10);
    }}
    .step-card:hover {{ transform: translateY(-4px); box-shadow: 0 14px 30px rgba(217, 139, 163, 0.22); border-color: {COLORS['secondary']}; }}
    .step-card.active {{ background: {COLORS['lavender']}; border-color: {COLORS['accent']}; }}
    .step-num {{
        width: 38px; height: 38px; margin: 0 auto 12px; border-radius: 50%; background: {COLORS['primary']};
        display: flex; align-items: center; justify-content: center; font-weight: 700; color: {COLORS['text_main']};
    }}
    .step-card.active .step-num {{ background: {COLORS['accent']}; color: #FFFFFF; }}
    .step-card h4 {{ margin: 0 0 6px !important; font-size: 16px !important; }}
    .step-card p {{ margin: 0; color: {COLORS['text_muted']}; font-size: 14px; line-height: 1.5; }}

    /* Footer */
    .footer-note {{
        text-align: center; color: {COLORS['text_muted']}; font-size: 13px; margin-top: 3rem;
        padding-top: 1.2rem; border-top: 1px solid {COLORS['primary']};
    }}

    /* Gentle fade-in */
    @keyframes fadeUp {{ from {{ opacity: 0; transform: translateY(14px); }} to {{ opacity: 1; transform: translateY(0); }} }}
    @media (prefers-reduced-motion: reduce) {{
        * {{ animation: none !important; transition: none !important; scroll-behavior: auto !important; }}
    }}
    @media (max-width: 900px) {{
        .nav-bar {{ padding: 0 1rem; }} .nav-links {{ gap: 0.4rem; }}
        .hero h1 {{ font-size: 2rem !important; }} .block-container {{ padding: 5.5rem 1rem 2rem !important; }}
    }}
</style>
""", unsafe_allow_html=True)

# ── 3. Database Initialization & Pre-Seeding ──
def get_db(): return sqlite3.connect("users_otp.db", check_same_thread=False)
def hash_txt(t): return bcrypt.hashpw(t.encode(), bcrypt.gensalt()).decode()
def check_txt(t, h): return bcrypt.checkpw(t.encode(), h.encode()) if h else False

with get_db() as conn:
    conn.execute("CREATE TABLE IF NOT EXISTS users (email TEXT PRIMARY KEY, password_hash TEXT)")
    for email in ["ranamehak570@gmail.com"]:
        if not conn.execute("SELECT email FROM users WHERE email=?", (email,)).fetchone():
            conn.execute("INSERT INTO users VALUES (?, ?)", (email, hash_txt("Welcome@123")))

# ── 4. Helper Functions (Secrets OTP & JWT Token) ──
def generate_otp(): return f"{secrets.randbelow(900000) + 100000}"

def is_strong_password(pw, email=""):
    if len(pw) < 10: return False, "Password must be at least 10 characters long."
    if " " in pw: return False, "Password must not contain spaces."
    if not re.search(r"[A-Z]", pw): return False, "Add at least one uppercase letter."
    if not re.search(r"[a-z]", pw): return False, "Add at least one lowercase letter."
    if not re.search(r"\d", pw): return False, "Add at least one number."
    if not re.search(r"[^A-Za-z0-9]", pw): return False, "Add at least one special character (e.g. @, #, $, !)."
    if email and email.split("@")[0].lower() in pw.lower(): return False, "Password must not contain your email name."
    if pw.lower() in ["password@123", "welcome@123", "admin@123", "qwerty@123"]: return False, "This password is too common."
    return True, "Strong"

def make_otp_token(email, otp):
    payload = {"sub": email, "otp_hash": hash_txt(otp), "type": "password_reset_otp", "iat": datetime.datetime.utcnow(), "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=OTP_EXPIRY_MINUTES)}
    return jwt.encode(payload, JWT_SECRET, algorithm="HS256")

def verify_otp_token(token, input_otp, email):
    try:
        payload = jwt.decode(token, JWT_SECRET, algorithms=["HS256"])
        if payload.get("sub") != email or payload.get("type") != "password_reset_otp": return False, "Security token mismatch."
        if check_txt(input_otp, payload["otp_hash"]): return True, "Valid"
        return False, "Invalid 6-digit OTP code."
    except jwt.ExpiredSignatureError: return False, f"⚠️ This OTP code expired after {OTP_EXPIRY_MINUTES} minutes. Please request a new one."
    except Exception as e: return False, "Invalid or corrupted verification token."

def send_professional_email(to_email, otp, app_pass):
    # 🚀 Anti-Spam Setup: Use MIMEMultipart('alternative') and standard RFC headers
    msg = MIMEMultipart('alternative')
    msg['From'] = f"Infosys Support <{SENDER_EMAIL}>"
    msg['To'] = to_email
    msg['Subject'] = "Infosys Portal - Verification Code"
    msg['Date'] = formatdate(localtime=True)  # Adds RFC compliant Date header
    msg['Message-ID'] = make_msgid()          # Prevents spam filtering
    msg['Reply-To'] = SENDER_EMAIL

    # Plain text version (Essential to bypass Gmail spam filters!)
    text_body = f"Your verification code for Infosys Portal is: {otp}\nThis code will expire in {OTP_EXPIRY_MINUTES} minutes.\nIf you did not request this code, please ignore this email."

    # Clean HTML version in the same soft pink palette
    html_body = f"""
    <!DOCTYPE html>
    <html>
    <head>
        <style>
            body {{ font-family: Arial, sans-serif; background-color: #FFFAFC; margin: 0; padding: 20px; }}
            .container {{ max-width: 500px; margin: 0 auto; background-color: #FFFFFF; border: 1.5px solid #F4B6C2; border-radius: 16px; padding: 30px; text-align: center; }}
            .title {{ color: #3D3036; font-size: 20px; font-weight: bold; margin-bottom: 15px; }}
            .text {{ color: #5E4D55; font-size: 15px; line-height: 1.5; margin-bottom: 20px; }}
            .otp-box {{ background-color: #F8C8DC; color: #3D3036; font-size: 28px; font-weight: bold; letter-spacing: 5px; padding: 15px 20px; border: 1.5px solid #D98BA3; border-radius: 12px; display: inline-block; margin: 10px 0; }}
            .footer {{ color: #7A6770; font-size: 12px; margin-top: 25px; border-top: 1px solid #EDE4F7; padding-top: 15px; }}
        </style>
    </head>
    <body>
        <div class="container">
            <div class="title">Infosys Portal Verification</div>
            <div class="text">We received a request to reset your password for <b>{to_email}</b>. Please use the verification code below:</div>
            <div class="otp-box">{otp}</div>
            <div class="text">This code expires in <b>{OTP_EXPIRY_MINUTES} minutes</b>.</div>
            <div class="footer">If you did not request this code, you can safely ignore this email.<br>&copy; 2026 Infosys Portal.</div>
        </div>
    </body>
    </html>
    """
    msg.attach(MIMEText(text_body, 'plain'))
    msg.attach(MIMEText(html_body, 'html'))
    try:
        clean_pass = (app_pass or "").replace(" ", "").strip()
        with smtplib.SMTP_SSL('smtp.gmail.com', 465, timeout=30) as s:
            s.login(SENDER_EMAIL, clean_pass)
            s.sendmail(SENDER_EMAIL, to_email, msg.as_string())
        return True, "Email sent successfully!"
    except smtplib.SMTPAuthenticationError:
        return False, "Gmail rejected the login. Check that EMAIL_PASSWORD is an App Password for ranamehak570@gmail.com."
    except Exception as e:
        return False, f"SMTP Error: {str(e)}"

# ── 5. Session State Management ──
for k, v in [("stage", "email"), ("reset_email", ""), ("jwt_token", "")]:
    if k not in st.session_state: st.session_state[k] = v

def restart_flow(): st.session_state.stage = "email"; st.session_state.reset_email = ""; st.rerun()

# ── 6. Page Layout ──
# Navigation bar
st.markdown("""
<div class="nav-bar">
    <div class="nav-brand">🌸 Infosys Portal</div>
    <div class="nav-links">
        <a href="#home">Home</a>
        <a href="#how-it-works">How it works</a>
        <a href="#security">Security</a>
    </div>
</div>
""", unsafe_allow_html=True)

# Hero
st.markdown(f"""
<div class="hero" id="home">
    <div class="hero-badge">🔐 Secure &amp; simple</div>
    <h1>Secure Password Reset</h1>
    <p>Verify your identity with a 6-digit OTP code sent to your email. It expires in {OTP_EXPIRY_MINUTES} minutes.</p>
</div>
""", unsafe_allow_html=True)

_, col_center, _ = st.columns([1, 2, 1])
with col_center:
    with st.container(border=True):

        # ── STAGE 1: ASK EMAIL ──
        if st.session_state.stage == "email":
            st.markdown("<h3 style='margin-top:0;font-size:19px;text-align:center;'>Enter your email ID</h3>", unsafe_allow_html=True)
            email = st.text_input("Email Address", placeholder="Enter your email ID", label_visibility="collapsed").lower().strip()
            st.markdown("<br>", unsafe_allow_html=True)

            if st.button("Send Verification Code →", use_container_width=True):
                if not email: st.error("⚠️ Please enter an email address.")
                elif not EMAIL_PASSWORD:
                    st.error("❌ Gmail App Password not found! Please add `EMAIL_PASSWORD` to your Colab Secrets tab.")
                else:
                    with get_db() as c: r = c.execute("SELECT 1 FROM users WHERE email=?", (email,)).fetchone()
                    if r:
                        otp = generate_otp()
                        with st.spinner(f"Sending 6-digit OTP (expires in {OTP_EXPIRY_MINUTES} mins)..."):
                            ok, msg = send_professional_email(email, otp, EMAIL_PASSWORD)
                        if ok:
                            st.session_state.reset_email = email
                            st.session_state.jwt_token = make_otp_token(email, otp)
                            st.session_state.stage = "otp"
                            st.success("✅ 6-digit OTP sent! Check your inbox."); time.sleep(1); st.rerun()
                        else: st.error(f"❌ {msg}")
                    else: st.error("❌ Email address not registered in the system.")

        # ── STAGE 2: ENTER OTP ──
        elif st.session_state.stage == "otp":
            st.markdown("<h3 style='margin-top:0;font-size:19px;text-align:center;'>Enter 6-Digit OTP Code</h3>", unsafe_allow_html=True)
            st.info(f"📧 Code sent to **{st.session_state.reset_email}** (Valid for {OTP_EXPIRY_MINUTES} mins).")
            otp_input = st.text_input("6-Digit Verification Code", placeholder="e.g. 849201", max_chars=6, label_visibility="collapsed")
            st.markdown("<br>", unsafe_allow_html=True)

            c1, c2 = st.columns([1.2, 1])
            if c1.button("Verify Code →", use_container_width=True):
                if not otp_input or len(otp_input) != 6: st.error("⚠️ Please enter the valid 6-digit code.")
                else:
                    ok, msg = verify_otp_token(st.session_state.jwt_token, otp_input, st.session_state.reset_email)
                    if ok:
                        st.session_state.stage = "reset"
                        st.success("✅ Code verified successfully!"); time.sleep(1); st.rerun()
                    else: st.error(f"❌ {msg}")
            if c2.button("← Back", use_container_width=True): restart_flow()

        # ── STAGE 3: NEW PASSWORD ──
        elif st.session_state.stage == "reset":
            st.markdown("<h3 style='margin-top:0;font-size:19px;text-align:center;'>Create New Password</h3>", unsafe_allow_html=True)
            npw = st.text_input("New Password (min 10 chars, A-z, 0-9, symbol)", type="password", placeholder="New Strong Password")
            confirm_npw = st.text_input("Confirm New Password", type="password", placeholder="Confirm New Password")
            st.markdown("<br>", unsafe_allow_html=True)

            c1, c2 = st.columns([1.2, 1])
            if c1.button("Update Password →", use_container_width=True):
                ok_pw, pw_msg = is_strong_password(npw, st.session_state.reset_email)
                if not ok_pw: st.error(f"⚠️ {pw_msg}")
                elif npw != confirm_npw: st.error("❌ Passwords do not match.")
                else:
                    with get_db() as c: c.execute("UPDATE users SET password_hash=? WHERE email=?", (hash_txt(npw), st.session_state.reset_email))
                    st.success("🎉 Password updated successfully! Your account is secure.")
                    time.sleep(2)
                    restart_flow()
            if c2.button("← Cancel", use_container_width=True): restart_flow()

# ── 7. How it works (highlights the current step) ──
stage_index = {"email": 0, "otp": 1, "reset": 2}[st.session_state.stage]
steps = [
    ("1", "Verify your email", "Enter your registered email and we'll send a 6-digit code."),
    ("2", "Enter the OTP", f"Type the code from your inbox. It stays valid for {OTP_EXPIRY_MINUTES} minutes."),
    ("3", "Set a new password", "Choose a strong password with letters, numbers and a symbol."),
]
cards = "".join(
    f'<div class="step-card{" active" if i == stage_index else ""}"><div class="step-num">{n}</div><h4>{t}</h4><p>{d}</p></div>'
    for i, (n, t, d) in enumerate(steps)
)
st.markdown(f"""
<div id="how-it-works" class="section-title">How it works</div>
<div class="steps">{cards}</div>
<div id="security" class="footer-note">🔒 Codes expire automatically and passwords are stored securely hashed.<br>&copy; 2026 Infosys Portal</div>
""", unsafe_allow_html=True)

Writing app.py


In [4]:
import os, time, subprocess
from pyngrok import ngrok
from google.colab import userdata

# 1. Safely retrieve secrets from Colab
try:
    os.environ['EMAIL_PASSWORD'] = userdata.get('EMAIL_PASSWORD')
    os.environ['JWT_SECRET'] = "super-secure-otp-secret-key-2026"
    ngrok_token = userdata.get('NGROK_AUTHTOKEN')
    if ngrok_token: ngrok.set_auth_token(ngrok_token)
except Exception as e:
    print(f"⚠️ Secret Error: Make sure EMAIL_PASSWORD and NGROK_AUTHTOKEN exist in Colab Secrets.")

# 🚀 2. Kill ONLY old Streamlit servers (Do NOT kill Ngrok so we don't lock your domain!)
!pkill -f streamlit
time.sleep(2)

# 3. Start Streamlit server cleanly in the background with new UI
process = subprocess.Popen(['streamlit', 'run', 'app.py', '--server.port', '8501'], env=os.environ.copy(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)

# 🚀 4. Safely Reuse or Connect Tunnel
try:
    tunnels = ngrok.get_tunnels()
    if tunnels:
        public_url = tunnels[0].public_url
        print("♻️ Reusing existing active Ngrok tunnel...")
    else:
        public_url = ngrok.connect(8501).public_url
        print("🚀 Created new Ngrok tunnel...")

    print("=" * 65)
    print(f"👉 OTP Password Reset Portal Live URL: {public_url}")
    print("=" * 65)
    print("⏳ Server running! Press [Ctrl + C] or click the Colab Stop button to shut down.")
    while True: time.sleep(1)

except Exception as e:
    print(f"⚠️ Notice: {e}")
    print("🔄 Waiting 5 seconds for Ngrok cloud servers to release domain lock...")
    ngrok.kill()
    !pkill -f ngrok
    time.sleep(5)  # 🚀 Give Ngrok cloud servers time to release your static endpoint!
    public_url = ngrok.connect(8501).public_url
    print("=" * 65)
    print(f"👉 OTP Password Reset Portal Live URL: {public_url}")
    print("=" * 65)
    while True: time.sleep(1)
except KeyboardInterrupt:
    print("\n🛑 Shutting down server...")
    !pkill -f streamlit
    print("✅ Streamlit stopped. (Ngrok kept alive for faster re-runs)")

🚀 Created new Ngrok tunnel...
👉 OTP Password Reset Portal Live URL: https://protector-runner-neatness.ngrok-free.dev
⏳ Server running! Press [Ctrl + C] or click the Colab Stop button to shut down.

🛑 Shutting down server...
✅ Streamlit stopped. (Ngrok kept alive for faster re-runs)
